# Fill-in values for Section 4 (no model training)

Reads existing result JSON only. Run all cells once, top to bottom (Runtime → Run all).

1. BERT 25 m/s: SD per cell and corrected decomposition (C, D at MAX_LEN=128)
2. Table 2: corrected 95% intervals for the mean accuracy of cells B and D, with comparison to the published values
3. Table 8: macro-F1 ± SD for the JamShield runs
4. Power: minimum detectable effect (80% power, alpha 0.05) from the corrected SE, macro-F1, 15 m/s

Requires `pcmef_fixed.py` and `pcmef_corrected_stats.py` in `Detector_Results/`. Outputs are saved to `Detector_Results/tahap_lanjutan/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil, sys, os, json, glob
import numpy as np, pandas as pd
from scipy import stats as sps

BASE = '/content/drive/MyDrive/Detector_Results'
for f in ('pcmef_fixed.py', 'pcmef_corrected_stats.py'):
    shutil.copy(f'{BASE}/{f}', f'/content/{f}')
if '/content' not in sys.path: sys.path.insert(0, '/content')
import pcmef_fixed as P
import pcmef_corrected_stats as CS

P15 = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_2.csv'
P25 = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_1.csv'
def load_and_filter(path, target_speed):
    df = pd.read_csv(path)
    df['label'] = df['Scenario'].map({1: 0, 2: 1, 3: 2})
    df = df.sort_values(['Speed', 'Scenario', 'Time']).reset_index(drop=True)
    df['pseudo_run_id'] = (df['Speed'].round(2).astype(str) + '_' + df['Scenario'].astype(str) + '_' +
                           (df['Time'] // 60).astype(int).astype(str))
    lo, hi = {15: (13, 17), 25: (23, 27)}[target_speed]
    return df[(df['Speed'] >= lo) & (df['Speed'] <= hi)].copy()

dfs15, _, d15 = P.filter_feasible_episodes(load_and_filter(P15, 15), min_duration_sec=15)
dfs25, _, d25 = P.filter_feasible_episodes(load_and_filter(P25, 25), min_duration_sec=15)
print(f'15 m/s dropped {d15} | 25 m/s dropped {d25}  (expected 130 / 115)')

OUT = f'{BASE}/factorial'
OUT_PILOT = f'{BASE}/pilot_bert_maxlen288'
OUT_NEXT = f'{BASE}/tahap_lanjutan'
os.makedirs(OUT_NEXT, exist_ok=True)
r15 = CS.episode_ratio(dfs15, P)
r25 = CS.episode_ratio(dfs25, P)
print('episode ratio 15 / 25 m/s:', r15, r25)

In [ ]:
PUBLISHED = {('RF_Stat', 15): 80.04, ('RF_Stat', 25): 94.61,
             ('LSTM', 15): 85.17, ('LSTM', 25): 95.67,
             ('BERT', 15): 90.25, ('BERT', 25): 96.25}   # accuracy in the original publication (verify 90.25 against the source)

def load_json(folder, model, sp):
    return [json.load(open(f)) for f in sorted(glob.glob(f'{folder}/{model}_{sp}ms_[ABCD]_seed*.json'))]

def cells(res, metric):
    """{cell: array of per-seed values (%)} for one metric."""
    out = {}
    for r in res:
        out.setdefault(r['cell'], {})[r['seed']] = r[metric] * 100
    return {c: np.array([v[s] for s in sorted(v)]) for c, v in out.items()}

def mean_ci_corrected(x, ratio, alpha=0.05):
    """Nadeau-Bengio style interval for a cell mean: SE = s * sqrt(1/J + ratio)."""
    x = np.asarray(x, float); J = len(x)
    se = x.std(ddof=1) * np.sqrt(1.0 / J + ratio)
    tc = sps.t.ppf(1 - alpha / 2, J - 1)
    m = x.mean()
    return m, x.std(ddof=1), max(m - tc * se, 0.0), min(m + tc * se, 100.0)

def mde(se, df, alpha=0.05, power=0.80):
    """Minimum detectable effect (two-sided) for a given SE and degrees of freedom."""
    return (sps.t.ppf(1 - alpha / 2, df) + sps.t.ppf(power, df)) * se

def effect_se(e):
    """SE and df of one corrected effect; falls back to the CI width if 'se_corr' is not stored."""
    df_ = e.get('df', 19)
    if 'se_corr' in e:
        return float(e['se_corr']), float(df_)
    for k in ('ci95_corr', 'ci95', 'ci_corr'):
        if k in e:
            lo, hi = e[k]
            return (hi - lo) / (2 * sps.t.ppf(0.975, df_)), float(df_)
    raise KeyError(f'no SE/CI key in effect dict; keys = {list(e)}')

In [ ]:
data = {}
for model in ('RF_Stat', 'LSTM', 'BERT'):
    for sp in (15, 25):
        res = load_json(OUT, model, sp)
        if model == 'BERT' and sp == 15:   # final configuration: C, D at MAX_LEN=288 from the pilot folder
            res = ([r for r in res if r['cell'] in ('A', 'B')] +
                   [r for r in load_json(OUT_PILOT, 'BERT', 15) if r['cell'] in ('C', 'D')])
        n = pd.Series([r['cell'] for r in res]).value_counts().sort_index().to_dict()
        print(f'{model} {sp} m/s: seeds per cell = {n}')
        data[(model, sp)] = res

In [ ]:
# Item 1: BERT 25 m/s (C, D at MAX_LEN=128)
res = data[('BERT', 25)]
for met in ('f1_macro', 'balanced_accuracy', 'accuracy'):
    v = cells(res, met)
    print(f'{met}: ' + ' | '.join(f'{c} {v[c].mean():.2f} ± {v[c].std(ddof=1):.2f}' for c in sorted(v)))
    CS.print_corrected(CS.decompose_factorial_corrected(res, met, ratio_pcmef=r25), f'BERT 25 m/s {met}')

In [ ]:
# Item 2: Table 2, corrected 95% intervals for cells B and D (PCMEF ratio)
rows = []
for (model, sp), res in data.items():
    v = cells(res, 'accuracy')
    r = r15 if sp == 15 else r25
    rec = dict(detector=model, regime=sp, published=PUBLISHED[(model, sp)])
    for c in ('B', 'D'):
        m, s, lo, hi = mean_ci_corrected(v[c], r)
        rec[f'{c}_mean'], rec[f'{c}_sd'] = round(m, 2), round(s, 2)
        rec[f'{c}_ci'] = f'[{lo:.1f}, {hi:.1f}]'
        rec[f'{c}_minus_pub'] = round(m - PUBLISHED[(model, sp)], 2)
        rec[f'{c}_contains_pub'] = bool(lo <= PUBLISHED[(model, sp)] <= hi)
    rows.append(rec)
t2 = pd.DataFrame(rows)
print(t2.to_string(index=False))
t2.to_csv(f'{OUT_NEXT}/table2_corrected_intervals.csv', index=False)

In [ ]:
# Item 4: minimum detectable effect (80% power, alpha 0.05), macro-F1, 15 m/s
rows = []
for model in ('RF_Stat', 'LSTM', 'BERT'):
    sm = CS.decompose_factorial_corrected(data[(model, 15)], 'f1_macro', ratio_pcmef=r15)
    for k in ('delta_protocol', 'delta_model', 'interaction'):
        e = sm[k]
        se, df_ = effect_se(e)
        rows.append(dict(model=model, effect=k, delta=round(e['mean_pp'], 2), se_corr=round(se, 2),
                         df=round(df_, 1), mde_80=round(mde(se, df_), 1)))
t4 = pd.DataFrame(rows)
print(t4.to_string(index=False))
t4.to_csv(f'{OUT_NEXT}/power_15ms.csv', index=False)

In [ ]:
# Item 3: JamShield macro-F1 ± SD per result folder (Table 8). Use the folder results_e510_v3_manual for the grouped 4-fold row.
JS = '/content/drive/MyDrive/JamShield_Dataset'
groups = {}
for f in glob.glob(f'{JS}/**/*.json', recursive=True):
    try: d = json.load(open(f))
    except Exception: continue
    if isinstance(d, dict) and 'f1_macro' in d:
        groups.setdefault(os.path.dirname(f), []).append(d)
sd = lambda a: a.std(ddof=1) if len(a) > 1 else float('nan')
rows = []
for folder, ds in sorted(groups.items()):
    f1 = np.array([d['f1_macro'] for d in ds]) * 100
    ba = np.array([d.get('balanced_accuracy', np.nan) for d in ds]) * 100
    rows.append(dict(folder=folder.replace(JS + '/', ''), n=len(ds), macroF1=f'{f1.mean():.2f} ± {sd(f1):.2f}',
                     balanced_acc=f'{np.nanmean(ba):.2f} ± {sd(ba):.2f}'))
js = pd.DataFrame(rows); print(js.to_string(index=False))
js.to_csv(f'{OUT_NEXT}/jamshield_f1_summary.csv', index=False)